# Project 2 — Retail Inventory Analytics

## Goal
This project analyzes a retail inventory dataset end to end:

1. clean the raw data in **Python / Pandas**  
2. load the cleaned table into **SQL Server**  
3. calculate KPIs and create views in **SQL**  
4. build an executive dashboard in **Power BI**

The notebook below is rewritten as a portfolio-ready version with proper EDA structure, explanations, and reusable code.

## 1) Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Optional, but useful for correlation heatmaps and styled plots
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2) Load the raw dataset

Use the raw CSV first, inspect the structure, and then create a cleaned version after validation.

In [ ]:
df = pd.read_csv("retail_store_inventory(1).csv")
df.head()

## 3) Initial data audit

Before cleaning, check:
- shape
- column names
- data types
- missing values
- duplicate rows
- basic descriptive statistics

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:\n")
print(df.dtypes)
print("\nNumeric summary:\n")
display(df.describe())

## 4) Data cleaning

### What should be fixed
- Convert `Date` to datetime
- Remove duplicate rows properly
- Check for invalid values in numeric columns
- Standardize column handling before export

The original notebook called `drop_duplicates()` without assigning it back to the dataframe.  
That does **not** modify the dataframe, so the corrected approach is shown below.

In [ ]:
# Convert date column
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Properly remove duplicates
df = df.drop_duplicates().copy()

# Review possible data quality issues
negative_forecast_rows = df[df["Demand Forecast"] < 0]
zero_sales_rows = df[df["Units Sold"] == 0]

print("Rows with negative demand forecast:", len(negative_forecast_rows))
print("Rows with zero units sold:", len(zero_sales_rows))

# Save cleaned file
df.to_csv("retail_store_inventory_ready(1).csv", index=False)
print("Cleaned file saved successfully.")

## 5) Business understanding

This dataset represents retail inventory activity across:
- multiple stores
- multiple product categories
- multiple regions
- daily observations over time

### Main business questions
- Which categories and regions contribute the most sales?
- How does promotion affect sales?
- Which products are understocked or overstocked?
- How do weather and seasonality influence demand?
- What is the forecast accuracy gap?

In [ ]:
# High-level summary
summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Date start",
        "Date end",
        "Stores",
        "Products",
        "Categories",
        "Regions",
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        df["Date"].min().date(),
        df["Date"].max().date(),
        df["Store ID"].nunique(),
        df["Product ID"].nunique(),
        df["Category"].nunique(),
        df["Region"].nunique(),
    ]
})
summary

## 6) Univariate analysis

Check the distribution of key columns like:
- Inventory Level
- Units Sold
- Units Ordered
- Demand Forecast
- Price
- Discount

In [ ]:
num_cols = ["Inventory Level", "Units Sold", "Units Ordered", "Demand Forecast", "Price", "Discount", "Competitor Pricing"]

df[num_cols].hist(figsize=(14, 8), bins=20)
plt.tight_layout()
plt.show()

## 7) Revenue and sales calculations

Create a few useful business metrics:
- total revenue
- total units sold
- average selling price
- inventory turnover ratio
- forecast accuracy

In [ ]:
df["Revenue"] = df["Units Sold"] * df["Price"]

total_revenue = df["Revenue"].sum()
total_units_sold = df["Units Sold"].sum()
avg_price = df["Price"].mean()
inventory_turnover_ratio = df["Units Sold"].sum() / df["Inventory Level"].mean()
forecast_accuracy = (1 - abs(df["Demand Forecast"].sum() - df["Units Sold"].sum()) / df["Demand Forecast"].sum()) * 100

metrics = pd.DataFrame({
    "Metric": [
        "Total revenue",
        "Total units sold",
        "Average price",
        "Inventory turnover ratio",
        "Forecast accuracy"
    ],
    "Value": [
        round(total_revenue, 2),
        int(total_units_sold),
        round(avg_price, 2),
        round(inventory_turnover_ratio, 2),
        round(forecast_accuracy, 2)
    ]
})

metrics

## 8) Category, region, season, and weather analysis

These views help identify where sales are coming from and which business factors matter most.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

df.groupby("Category")["Units Sold"].sum().sort_values(ascending=False).plot(kind="bar", ax=axes[0, 0], title="Units Sold by Category")
df.groupby("Region")["Units Sold"].sum().sort_values(ascending=False).plot(kind="bar", ax=axes[0, 1], title="Units Sold by Region")
df.groupby("Seasonality")["Units Sold"].sum().sort_values(ascending=False).plot(kind="bar", ax=axes[1, 0], title="Units Sold by Season")
df.groupby("Weather Condition")["Units Sold"].sum().sort_values(ascending=False).plot(kind="bar", ax=axes[1, 1], title="Units Sold by Weather")

for ax in axes.flat:
    ax.set_xlabel("")
    ax.set_ylabel("Units Sold")

plt.tight_layout()
plt.show()

## 9) Promotion and discount analysis

Use this section to understand whether promotional activity and discounts are linked to higher sales.

In [ ]:
promo = df.groupby("Holiday/Promotion")["Units Sold"].agg(["sum", "mean"]).reset_index()
promo.columns = ["Holiday/Promotion", "Total Units Sold", "Avg Units Sold"]
promo

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["Discount"], df["Units Sold"])
plt.title("Discount vs Units Sold")
plt.xlabel("Discount")
plt.ylabel("Units Sold")
plt.show()

## 10) Time-series analysis

A date-based trend helps show changes over time and supports executive reporting in Power BI.

In [ ]:
monthly = (
    df.set_index("Date")
      .resample("M")[["Units Sold", "Revenue"]]
      .sum()
      .reset_index()
)

monthly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly["Date"], monthly["Revenue"])
ax.set_title("Monthly Revenue Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
plt.tight_layout()
plt.show()

## 11) Correlation analysis

Check relationships between the numeric features to support hypothesis building.

In [ ]:
corr = df[["Inventory Level", "Units Sold", "Units Ordered", "Demand Forecast", "Price", "Discount", "Competitor Pricing"]].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 12) SQL handoff and Power BI handoff

After cleaning, the dataset is loaded into SQL Server to calculate KPIs and create reusable views, then into Power BI for dashboard development.

### SQL layer
Examples of outputs created in SQL:
- total revenue
- total units sold
- inventory turnover ratio
- stock coverage days
- forecast accuracy
- out-of-stock risk
- overstock analysis
- category and region analysis
- promotion, weather, seasonality, and discount analysis
- product performance and revenue trend views

### Power BI layer
The report contains four pages:
- Executive Overview
- Inventory Analytics
- Demand & Customer Analytics
- Product Performance

## 13) Final findings

Use your final narrative to explain:
- which category performed best
- which region led sales
- whether promotions improved sales
- where stock risk is highest
- how accurate the forecast is

This is the section recruiters usually care about most because it shows business thinking, not just tool usage.

## 14) Export cleaned data

If you make changes to the cleaning logic, re-run the notebook and export the final cleaned file again.

In [ ]:
df.to_csv("retail_store_inventory_ready(1).csv", index=False)